##Step 1: Setup and Environment

###1. Initial Installations

In [1]:
# Install necessary libraries for our project
!pip install transformers datasets torch accelerate huggingface_hub

Defaulting to user installation because normal site-packages is not writeable


###2. Authenticating with Hugging Face

In [2]:
# Log in to your Hugging Face account to access the gated Llama model
from huggingface_hub import login
login()

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


In [4]:
import torch
print(torch.cuda.is_available())

True


Model Loading & Tokeninzing

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token_id = tokenizer.eos_token_id

print("Loading model with SDPA attention for A100 compatibility...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    # Use PyTorch's native attention, which works without extra installation
    attn_implementation="sdpa",
)

print("-------------------------------------------")
print(f"Model successfully loaded on device: {model.device}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading model with SDPA attention for A100 compatibility...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

-------------------------------------------
Model successfully loaded on device: cuda:0


##Step 2: Baseline 1 - Zero-Shot Chain-of-Thought (CoT) Evaluation

Our script will perform the following actions:

Load the GSM8k test dataset.

Loop through each question in the test set.

Format each question into a Zero-Shot CoT prompt using the special chat template for Llama 3.2.

Use our loaded model to generate a step-by-step answer.

Extract the final numerical answer from the model's text response.

Compare the extracted answer to the correct answer and calculate the overall accuracy.

In [ ]:
import re
import datasets
from tqdm import tqdm
import torch

# Helper function
def extract_final_answer(model_output):
    numbers = re.findall(r'[\d,]+\.?\d*', model_output)
    if numbers:
        return numbers[-1].replace(',', '')
    return None

# Load the evaluation dataset
print("Loading GSM8k test dataset...")
gsm8k_test_dataset = datasets.load_dataset("openai/gsm8k", 'main')['test']
print(f"Dataset loaded. Number of test examples: {len(gsm8k_test_dataset)}")

# Main evaluation loop
correct_predictions = 0
total_predictions = 0

for example in tqdm(gsm8k_test_dataset, desc="Evaluating Zero-Shot CoT"):
    question = example['question']
    ground_truth_answer = extract_final_answer(example['answer'])
    prompt = f"{question}\n\nLet's think step by step."
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_text = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
    model_answer = extract_final_answer(response_text)

    # Check if both strings are valid (not None and not empty) before converting to float
    if model_answer and ground_truth_answer:
        try:
            if float(model_answer) == float(ground_truth_answer):
                correct_predictions += 1
        except ValueError:
            # Handles any other weird cases where the extracted string is still not a valid number
            pass
    total_predictions += 1

# Calculate the final accuracy
accuracy = (correct_predictions / total_predictions) * 100
print("\n--- Zero-Shot CoT Evaluation Complete ---")
print(f"Correct Predictions: {correct_predictions}")
print(f"Total Predictions: {total_predictions}")
print(f"Accuracy: {accuracy:.2f}%")

# Store the result for our final report
zero_shot_cot_accuracy = accuracy

Loading GSM8k test dataset...
Dataset loaded. Number of test examples: 1319


Evaluating Zero-Shot CoT: 100%|██████████| 1319/1319 [3:02:19<00:00,  8.29s/it]


--- Zero-Shot CoT Evaluation Complete ---
Correct Predictions: 844
Total Predictions: 1319
Accuracy: 63.99%


##Step 3: Baseline 2 - Vanilla Supervised Fine-Tuning.

The Plan
Prepare the Data: We'll write a function to format each question and answer pair from the GSM8k train set into a single sequence for the model to learn from.

Configure Training: We will use the train_config class from your professor's notebook to set parameters like batch size, learning rate, and number of epochs.

Main train, evaluation, and helper functions from the notebook you provided.

Run Fine-Tuning: We will kick off the training process. This will take some time.

Evaluate: After training is complete, we'll load our new, fine-tuned model and evaluate it on the test set to get our Baseline 2 accuracy score.

1. Data Preparation

In [6]:
import datasets

def get_preprocessed_dataset_sft(tokenizer, split):
    """
    Loads and preprocesses the GSM8k dataset for supervised fine-tuning.
    """
    dataset = datasets.load_dataset("openai/gsm8k", 'main')[split]

    def format_prompt(sample):
        # Create the instruction prompt
        prompt = f"### Instruction:\n{sample['question']}\n\n### Response:\n"

        # Combine with the ground-truth response, including the end-of-sequence token
        response = f"{sample['answer']}{tokenizer.eos_token}"

        # Tokenize and format for training
        formatted_sample = tokenizer(prompt + response)

        # The 'labels' are the same as 'input_ids' for auto-regressive language modeling
        formatted_sample["labels"] = formatted_sample["input_ids"].copy()

        return formatted_sample

    # Apply the formatting to the entire dataset
    processed_dataset = dataset.map(format_prompt, remove_columns=list(dataset.features))
    return processed_dataset

2. Training Configuration

In [14]:
from dataclasses import dataclass

@dataclass
class SFT_Config:
    model_name: str = "meta-llama/Llama-3.2-3B-Instruct"
    dataset_name: str = "openai/gsm8k"
    num_epochs: int = 1
    batch_size_training: int = 16
    gradient_accumulation_steps: int = 1
    context_length: int = 512
    num_workers_dataloader: int = 1
    lr: float = 2e-5
    batching_strategy: str = "packing"
    run_validation: bool = True
    val_batch_size: int = 16
    output_dir: str = "./gsm8k_sft_model"
    save_model: bool = True
    gradient_clipping: bool = True
    gradient_clipping_threshold: float = 1.0
    weight_decay: float = 0.1
    gamma: float = 0.85
    seed: int = 42
    max_train_step: int = 0
    max_eval_step: int = 0
    save_metrics: bool = False

# Instantiate the config
train_config = SFT_Config()

3. Training and Helper Functions

In [15]:
# This cell contains the training infrastructure provided in the professor's notebook.
# No changes are needed here. Just run it.

from torch.utils.data import Dataset
from tqdm import tqdm
from transformers import default_data_collator, DataCollatorForSeq2Seq
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from contextlib import nullcontext
import time
import os
import numpy as np
import random


class ConcatDataset(Dataset):
    def __init__(self, dataset, chunk_size=4096):
        self.dataset = dataset
        self.chunk_size = chunk_size
        self.samples = []
        buffer = {"input_ids": [], "attention_mask": [], "labels": []}
        for sample in tqdm(self.dataset, desc="Packing dataset"):
            buffer = {k: v + sample[k] for k, v in buffer.items()}
            while len(next(iter(buffer.values()))) > self.chunk_size:
                self.samples.append({k: v[:self.chunk_size] for k, v in buffer.items()})
                buffer = {k: v[self.chunk_size:] for k, v in buffer.items()}

    def __getitem__(self, idx):
        return self.samples[idx]

    def __len__(self):
        return len(self.samples)

def train(model, train_dataloader, eval_dataloader, tokenizer, optimizer, lr_scheduler, gradient_accumulation_steps, train_config, wandb_run=None):
    results = {}
    best_val_loss = float("inf")
    for epoch in range(train_config.num_epochs):
        model.train()
        total_loss = 0.0
        pbar = tqdm(colour="blue", desc=f"Fine-Tuning Epoch: {epoch+1}", total=len(train_dataloader), dynamic_ncols=True)
        for step, batch in enumerate(train_dataloader):
            for key in batch.keys():
                batch[key] = batch[key].to(model.device)
            with nullcontext():
                outputs = model(**batch)
                loss = outputs.loss
            loss = loss / gradient_accumulation_steps
            total_loss += loss.detach().float()
            loss.backward()
            if (step + 1) % gradient_accumulation_steps == 0 or step == len(train_dataloader) - 1:
                if train_config.gradient_clipping:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), train_config.gradient_clipping_threshold)
                optimizer.step()
                optimizer.zero_grad()
                pbar.update(1)
            pbar.set_description(f"Epoch {epoch+1}, Loss: {loss.detach().float():.4f}")

        pbar.close()
        train_epoch_loss = total_loss / len(train_dataloader)
        train_perplexity = torch.exp(train_epoch_loss)
        print(f"Epoch {epoch+1}: Train Perplexity: {train_perplexity:.4f}, Train Loss: {train_epoch_loss:.4f}")

        if train_config.run_validation:
            eval_ppl, eval_epoch_loss = evaluation(model, train_config, eval_dataloader, tokenizer)
            if eval_epoch_loss < best_val_loss:
                best_val_loss = eval_epoch_loss
                if train_config.save_model:
                    print(f"New best validation loss: {best_val_loss:.4f}. Saving model...")
                    model.save_pretrained(train_config.output_dir)
                    tokenizer.save_pretrained(train_config.output_dir)
            val_loss = best_val_loss
            val_prep = eval_ppl
    results['avg_train_loss'] = train_epoch_loss
    if train_config.run_validation:
        results['avg_eval_loss'] = val_loss
    return results

def evaluation(model, train_config, eval_dataloader, tokenizer):
    model.eval()
    eval_loss = 0.0
    with torch.no_grad():
        for step, batch in enumerate(tqdm(eval_dataloader, colour="green", desc="Evaluating", dynamic_ncols=True)):
            for key in batch.keys():
                batch[key] = batch[key].to(model.device)
            outputs = model(**batch)
            loss = outputs.loss
            eval_loss += loss.detach().float()
    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    print(f"Validation Perplexity: {eval_ppl:.4f}, Validation Loss: {eval_epoch_loss:.4f}")
    return eval_ppl, eval_epoch_loss

4. Prepare Dataloaders and Run Training

In [16]:
# 1. Prepare datasets
full_train_dataset = get_preprocessed_dataset_sft(tokenizer, 'train')
full_eval_dataset = get_preprocessed_dataset_sft(tokenizer, 'test') # We'll use a slice of test for validation

# For validation during training, we'll just use a small subset of the test data
val_subset_size = 200
val_dataset = full_eval_dataset.select(range(val_subset_size))

# Pack the datasets
packed_train_dataset = ConcatDataset(full_train_dataset, chunk_size=train_config.context_length)
packed_val_dataset = ConcatDataset(val_dataset, chunk_size=train_config.context_length)

# 2. Create Dataloaders
train_dataloader = torch.utils.data.DataLoader(
    packed_train_dataset,
    batch_size=train_config.batch_size_training,
    num_workers=train_config.num_workers_dataloader,
    pin_memory=True,
    shuffle=True, # Shuffle for training
    collate_fn=default_data_collator,
    drop_last=True
)

eval_dataloader = torch.utils.data.DataLoader(
    packed_val_dataset,
    batch_size=train_config.val_batch_size,
    num_workers=train_config.num_workers_dataloader,
    pin_memory=True,
    shuffle=False, # No need to shuffle for validation
    collate_fn=default_data_collator,
    drop_last=True
)

# 3. Setup optimizer and scheduler
optimizer = optim.AdamW(model.parameters(), lr=train_config.lr, weight_decay=train_config.weight_decay)
scheduler = StepLR(optimizer, step_size=1, gamma=train_config.gamma)

# 4. Start Training!
print("--- Starting Supervised Fine-Tuning (SFT) ---")
results = train(
    model,
    train_dataloader,
    eval_dataloader,
    tokenizer,
    optimizer,
    scheduler,
    train_config.gradient_accumulation_steps,
    train_config,
)
print("--- SFT Complete ---")
print(results)

Packing dataset: 100%|██████████| 200/200 [00:00<00:00, 5243.47it/s]


--- Starting Supervised Fine-Tuning (SFT) ---


Fine-Tuning Epoch: 1:   0%|          | 0/151 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Epoch 1, Loss: 0.9237: 100%|██████████| 151/151 [02:32<00:00,  1.01s/it]


Epoch 1: Train Perplexity: 2.7670, Train Loss: 1.0178


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Evaluating: 100%|██████████| 4/4 [00:01<00:00,  3.72it/s]


Validation Perplexity: 2.8537, Validation Loss: 1.0486
New best validation loss: 1.0486. Saving model...
--- SFT Complete ---
{'avg_train_loss': tensor(1.0178, device='cuda:0'), 'avg_eval_loss': tensor(1.0486, device='cuda:0')}


5. Evaluate the Fine-Tuned Model

In [6]:
import re
from tqdm import tqdm
import torch
import datasets
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- Load the fine-tuned model from the directory we saved it to ---
print("Loading the fine-tuned SFT model for evaluation...")
sft_model_path = "./gsm8k_sft_model"
tokenizer = AutoTokenizer.from_pretrained(sft_model_path)
model = AutoModelForCausalLM.from_pretrained(
    sft_model_path,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    # THIS IS THE FIX: Use the compatible SDPA implementation
    attn_implementation="sdpa",
)
print("SFT model loaded.")

# --- We re-use the exact same evaluation logic from Step 2 ---
def extract_final_answer(model_output):
    numbers = re.findall(r'[\d,]+\.?\d*', model_output)
    if numbers:
        return numbers[-1].replace(',', '')
    return None

gsm8k_test_dataset = datasets.load_dataset("openai/gsm8k", 'main')['test']
correct_predictions = 0
total_predictions = 0

for example in tqdm(gsm8k_test_dataset, desc="Evaluating SFT Model"):
    question = example['question']
    ground_truth_answer = extract_final_answer(example['answer'])
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"

    # We use a simple prompt format now, not the chat template,
    # because we have fine-tuned the model to follow this specific new format.
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response_text = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    model_answer = extract_final_answer(response_text)

    if model_answer and ground_truth_answer:
        try:
            if float(model_answer) == float(ground_truth_answer):
                correct_predictions += 1
        except ValueError:
            pass
    total_predictions += 1

accuracy = (correct_predictions / total_predictions) * 100
print("\n--- SFT Model Evaluation Complete ---")
print(f"Correct Predictions: {correct_predictions}")
print(f"Total Predictions: {total_predictions}")
print(f"Accuracy: {accuracy:.2f}%")

# Store the result for our final report
sft_accuracy = accuracy

Loading the fine-tuned SFT model for evaluation...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

SFT model loaded.


Evaluating SFT Model: 100%|██████████| 1319/1319 [46:47<00:00,  2.13s/it]


--- SFT Model Evaluation Complete ---
Correct Predictions: 842
Total Predictions: 1319
Accuracy: 63.84%


##Step 4: The Main Event - Implementing STaR

###Part A: Create the Bootstrapped Dataset

The Plan

Load the original, pre-trained Llama 3.2 model (the one from Step 1, not the fine-tuned one from Step 3).

Iterate through every single example in the GSM8k train set.

First Attempt (Zero-Shot CoT): For each question, we'll ask the model to generate a rationale and answer.

Check the Answer:

If the model's final answer is CORRECT, we'll save that question and the model's successful rationale.

If the model's final answer is WRONG, we'll move to the next step.

Second Attempt (Rationalization): For the failed questions, we'll give the model a hint by providing the correct answer in the prompt. We'll ask it, "Can you explain the steps to get to this specific answer?" This is the "reasoning backward" trick from the paper. We'll then save that question and the new, rationalized explanation.

Save the Dataset: The final result will be a new dataset file containing all 7,473 questions, each paired with a high-quality rationale that we generated.

In [5]:
import re
import json
import datasets
from tqdm import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Reloading the model with the compatible attention implementation ---
print("Loading the original pre-trained model for STaR data generation...")
model_id = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    # THIS IS THE PERMANENT FIX for your environment
    attn_implementation="sdpa",
)

tokenizer.pad_token_id = tokenizer.eos_token_id
print("Original model loaded successfully.")

# --- Helper function from before ---
def extract_final_answer(model_output):
    numbers = re.findall(r'[\d,]+\.?\d*', model_output)
    if numbers:
        return numbers[-1].replace(',', '')
    return None

# --- Load the GSM8k training dataset ---
gsm8k_train_dataset = datasets.load_dataset("openai/gsm8k", 'main')['train']

# --- Main STaR Dataset Generation Loop ---
star_dataset = []
output_file_path = "./star_dataset.jsonl"

for example in tqdm(gsm8k_train_dataset, desc="Generating STaR Dataset"):
    question = example['question']
    ground_truth_rationale = example['answer']
    ground_truth_answer = extract_final_answer(ground_truth_rationale)

    # 1. First Attempt: Zero-Shot CoT
    zeroshot_prompt = f"{question}\n\nLet's think step by step."
    messages = [{"role": "user", "content": zeroshot_prompt}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(inputs, max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.eos_token_id)

    generated_text = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
    model_answer = extract_final_answer(generated_text)
    
    # Check if the answer is valid before converting to float
    is_correct = False
    if model_answer and ground_truth_answer:
        try:
            if float(model_answer) == float(ground_truth_answer):
                is_correct = True
        except ValueError:
            pass 

    if is_correct:
        final_rationale = generated_text
    else:
        # 3. If incorrect, use Rationalization
        rationalization_prompt = f"Question: {question}\nHere is the correct answer: {ground_truth_answer}\n\nPlease provide a step-by-step explanation of how to arrive at this answer."
        messages = [{"role": "user", "content": rationalization_prompt}]
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(inputs, max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        
        final_rationale = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)

    star_dataset.append({"question": question, "answer": final_rationale})

# 5. Save the final dataset
with open(output_file_path, 'w') as f:
    for entry in star_dataset:
        f.write(json.dumps(entry) + '\n')

print(f"\n--- STaR Dataset Generation Complete ---")
print(f"Dataset saved to: {output_file_path}")
print(f"Total examples generated: {len(star_dataset)}")

Loading the original pre-trained model for STaR data generation...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Original model loaded successfully.


Generating STaR Dataset:   0%|          | 0/7473 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Generating STaR Dataset: 100%|██████████| 7473/7473 [10:13:20<00:00,  4.92s/it]  


--- STaR Dataset Generation Complete ---
Dataset saved to: ./star_dataset.jsonl
Total examples generated: 7473


###Part B: Fine-Tuning with the STaR Dataset

1. Load and Prepare the STaR Dataset

In [7]:
import datasets
from datasets import load_dataset

def get_preprocessed_dataset_star(tokenizer, file_path):
    """
    Loads and preprocesses the STaR dataset from the generated JSONL file.
    """
    # Load the dataset from the local jsonl file
    dataset = load_dataset('json', data_files=file_path, split='train')

    def format_prompt_star(sample):
        # The prompt structure is the same as our SFT baseline
        prompt = f"### Instruction:\n{sample['question']}\n\n### Response:\n"
        response = f"{sample['answer']}{tokenizer.eos_token}"
        formatted_sample = tokenizer(prompt + response)
        formatted_sample["labels"] = formatted_sample["input_ids"].copy()
        return formatted_sample

    processed_dataset = dataset.map(format_prompt_star, remove_columns=list(dataset.features))
    return processed_dataset

2. STaR Training Configuration

In [8]:
from dataclasses import dataclass

@dataclass
class STaR_Config:
    # Model and dataset configs
    model_name: str = "meta-llama/Llama-3.2-3B-Instruct"
    # Note: We are now using our local dataset file
    dataset_file: str = "./star_dataset.jsonl"

    # Training parameters (kept the same as SFT for fair comparison)
    num_epochs: int = 1
    batch_size_training: int = 2
    gradient_accumulation_steps: int = 4
    lr: float = 2e-5

    # Dataloader and context length
    context_length: int = 512
    batching_strategy: str = "packing"
    num_workers_dataloader: int = 2

    # Validation and saving
    run_validation: bool = True
    val_batch_size: int = 2
    # Save to a new directory to keep it separate from the SFT model
    output_dir: str = "./star_model"
    save_model: bool = True

    # Other parameters
    gradient_clipping: bool = True
    gradient_clipping_threshold: float = 1.0
    weight_decay: float = 0.1
    gamma: float = 0.85
    seed: int = 42
    max_train_step: int = 0
    max_eval_step: int = 0
    save_metrics: bool = False

# Instantiate the config
train_config_star = STaR_Config()

3. Run STaR Fine-Tuning

In [10]:
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from transformers import AutoModelForCausalLM, AutoTokenizer, default_data_collator

# 1. IMPORTANT: Reload the original base model
print("Reloading the original base model for STaR fine-tuning...")
model_id = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    # THIS IS THE PERMANENT FIX for your environment
    attn_implementation="sdpa",
)
tokenizer.pad_token_id = tokenizer.eos_token_id
print("Original model reloaded.")

# 2. Prepare datasets using our new function and the STaR data file
full_star_dataset = get_preprocessed_dataset_star(tokenizer, train_config_star.dataset_file)

# We'll create a train/validation split from our new STaR dataset
train_val_split = full_star_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset_star = train_val_split['train']
val_dataset_star = train_val_split['test']

# Pack the datasets
packed_train_dataset_star = ConcatDataset(train_dataset_star, chunk_size=train_config_star.context_length)
packed_val_dataset_star = ConcatDataset(val_dataset_star, chunk_size=train_config_star.context_length)

# 3. Create Dataloaders
train_dataloader_star = torch.utils.data.DataLoader(
    packed_train_dataset_star,
    batch_size=train_config_star.batch_size_training,
    num_workers=train_config_star.num_workers_dataloader,
    pin_memory=True,
    shuffle=True,
    collate_fn=default_data_collator,
    drop_last=True
)

eval_dataloader_star = torch.utils.data.DataLoader(
    packed_val_dataset_star,
    batch_size=train_config_star.val_batch_size,
    num_workers=train_config_star.num_workers_dataloader,
    pin_memory=True,
    shuffle=False,
    collate_fn=default_data_collator,
    drop_last=True
)

# 4. Setup optimizer and scheduler
optimizer = optim.AdamW(model.parameters(), lr=train_config_star.lr, weight_decay=train_config_star.weight_decay)
scheduler = StepLR(optimizer, step_size=1, gamma=train_config_star.gamma)

# 5. Start Training!
print("--- Starting STaR Fine-Tuning ---")
results_star = train(
    model,
    train_dataloader_star,
    eval_dataloader_star,
    tokenizer,
    optimizer,
    scheduler,
    train_config_star.gradient_accumulation_steps,
    train_config_star, # Pass in the new STaR config
)
print("--- STaR SFT Complete ---")
print(results_star)

Reloading the original base model for STaR fine-tuning...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Original model reloaded.


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

NameError: name 'ConcatDataset' is not defined